In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import MACCSkeys
from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect
import numpy as np
import pandas as pd
import xgboost as xgb
from tqdm import tqdm
from sklearn.model_selection import train_test_split

   ---------------------------------------- 0.0/20.5 MB ? eta -:--:--
    --------------------------------------- 0.3/20.5 MB ? eta -:--:--
   --- ------------------------------------ 1.8/20.5 MB 8.4 MB/s eta 0:00:03
   ----- ---------------------------------- 2.9/20.5 MB 6.0 MB/s eta 0:00:03
   ------ --------------------------------- 3.4/20.5 MB 5.3 MB/s eta 0:00:04
   ------- -------------------------------- 3.7/20.5 MB 4.6 MB/s eta 0:00:04
   ------- -------------------------------- 3.9/20.5 MB 4.1 MB/s eta 0:00:05
   -------- ------------------------------- 4.5/20.5 MB 3.3 MB/s eta 0:00:05
   --------- ------------------------------ 4.7/20.5 MB 3.1 MB/s eta 0:00:06
   ---------- ----------------------------- 5.5/20.5 MB 2.8 MB/s eta 0:00:06
   ----------- ---------------------------- 6.0/20.5 MB 2.8 MB/s eta 0:00:06
   ------------ --------------------------- 6.6/20.5 MB 2.8 MB/s eta 0:00:05
   ------------- -------------------------- 7.1/20.5 MB 2.8 MB/s eta 0:00:05
   ----------

ERROR: Could not install packages due to an OSError: [WinError 5] Отказано в доступе: 'C:\\Users\\sypna\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python311\\site-packages\\rdkit\\rdBase.pyd'
Check the permissions.


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\sypna\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
def components(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    desc = [
        Descriptors.MolLogP(mol),                # Коэффициент распределения октанол/вода
        Descriptors.MolWt(mol),                   # Молекулярная масса
        Descriptors.NumHDonors(mol),              # Число доноров водородной связи
        Descriptors.NumHAcceptors(mol),           # Число акцепторов водородной связи
        Descriptors.TPSA(mol),                    # Топологическая полярная поверхность
        Descriptors.NumRotatableBonds(mol),       # Число вращающихся связей
        Descriptors.RingCount(mol),               # Общее число колец
        Descriptors.HeavyAtomCount(mol),          # Число тяжелых атомов
        Descriptors.NHOHCount(mol),                # Число групп NH и OH
        Descriptors.NOCount(mol),                  # Число атомов N и O
        Descriptors.NumValenceElectrons(mol),      # Число валентных электронов
        Descriptors.MaxPartialCharge(mol),         # Максимальный парциальный заряд
        Descriptors.MinPartialCharge(mol),         # Минимальный парциальный заряд
    ]
    return desc

In [2]:
def gen_fingetprints(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    fp=list(fp)
    maccs = MACCSkeys.GenMACCSKeys(mol)
    maccs = list(maccs)
    return fp+maccs

In [9]:
def unite_params(smiles1, smiles2):
    desc1 = components(smiles1)
    desc2 = components(smiles2)
    fp1 = gen_fingetprints(smiles1)
    fp2 = gen_fingetprints(smiles2)
    if desc1 is None or desc2 is None or fp1 is None or fp2 is None:
        return None
    diff = [desc2[i] - desc2[i] for i in range(len(desc1))]
    
    features = desc1 + desc2 + fp1 + fp2 + diff
    return features



In [ ]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
def build_dataset(df, has_target=True):
    X = []
    y = []
    for idx, row in tqdm(df.iterrows(), total=len(df), desc='Processing'):
        feat = unite_params(row['SMILES1'], row['SMILES2'])
        if feat is not None:
            X.append(feat)
            if has_target:
                y.append(row['result'])
    X = np.array(X)
    if has_target:
        y = np.array(y)
        return X, y
    else:
        return X

print("Processing train...")
X_train, y_train = build_dataset(df_train)
print("Processing test...")
X_test = build_dataset(df_test, has_target=False)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

In [13]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train)
neg = (y_tr == 0).sum()
pos = (y_tr == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {scale_pos_weight:.3f}")

scale_pos_weight = 0.129


In [18]:
model = xgb.XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_estimators=200,
    max_depth=6,
    learning_rate=0.9,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=20
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=False
)
preds = model.predict_proba(X_test)[:, 1]
submission = pd.DataFrame({
    'id': df_test['id'],
    'result': (preds > 0.5).astype(int)
})
submission.to_csv('submission4.csv', index=False)
print("Submission saved.")

C:\Users\sypna\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\xgboost\callback.py:385: UserWarning: [18:02:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


Submission saved.
